# Failure Analysis

Load test-ID and test-OOD datasets, plot the distribution of relative L2 errors per sample, and inspect the worst OOD failures. This notebook is meant to be run *after* training.

To keep this file small, actual code is minimal; expand as needed.

In [10]:
import os

# 1. Cloner le dépôt (si ce n'est pas déjà fait)
if not os.path.exists('schrodinger-fno'):
    !git clone https://github.com/NodeBox-Labs/schrodinger-fno.git

# 2. Se déplacer dans le dossier
%cd schrodinger-fno

# 3. Installer les dépendances manquantes
!pip install -q pyyaml tqdm pytest

# 4. Générer un petit jeu de données (rapide, juste pour tester le notebook)
if not os.path.exists('data/test_id.pt'):
    print("Génération des données...")
    !python scripts/generate_data.py --config configs/data.yaml --out data/

# 5. Entraîner le modèle (rapide, 1 seul époque pour éviter d'attendre)
if not os.path.exists('results/fno_single_seed42.pt'):
    print("Entraînement d'un modèle rapide...")
    !python scripts/train.py --config configs/model_fno.yaml --data data/ --mode single --seed 42

Cloning into 'schrodinger-fno'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 64 (delta 5), reused 62 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 31.32 KiB | 15.66 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/schrodinger-fno
Génération des données...
Traceback (most recent call last):
  File "/content/schrodinger-fno/scripts/generate_data.py", line 8, in <module>
    from src.data.potentials import make_potential
ModuleNotFoundError: No module named 'src'
Entraînement d'un modèle rapide...
Traceback (most recent call last):
  File "/content/schrodinger-fno/scripts/train.py", line 7, in <module>
    from src.train.loop import train_model
ModuleNotFoundError: No module named 'src'


In [11]:
import sys
import os

# Obtenir le répertoire de travail actuel (qui sera /content/schrodinger-fno)
project_root = os.getcwd()

# Si nous sommes dans le dossier notebooks, remonter d'un niveau
if project_root.endswith('notebooks'):
    project_root = os.path.abspath(os.path.join(project_root, '..'))

# Ajouter au sys.path pour que `import src` fonctionne
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root configuré sur : {project_root}")

Project root configuré sur : /content/schrodinger-fno


In [ ]:
def per_sample_errors(path):
    ds = SchrodingerDataset(path)
    errs = []
    with torch.no_grad():
        for i in range(len(ds)):
            inp, tgt = ds[i]
            pred = m(inp.unsqueeze(0).to(dev)).cpu().squeeze(0)
            errs.append(rel_l2(pred.unsqueeze(0), tgt.unsqueeze(0)).item())
    return np.array(errs)

# Ensure the data exists before running
import os
if os.path.exists('data/test_id.pt') and os.path.exists('data/test_ood.pt'):
    e_id = per_sample_errors('data/test_id.pt')
    e_ood = per_sample_errors('data/test_ood.pt')
    print(f'ID   median/p95/max: {np.median(e_id):.4e} / {np.percentile(e_id, 95):.4e} / {e_id.max():.4e}')
    print(f'OOD  median/p95/max: {np.median(e_ood):.4e} / {np.percentile(e_ood, 95):.4e} / {e_ood.max():.4e}')
else:
    print("Data files not found. Please run `python scripts/generate_data.py` first.")

Data files not found. Please run `python scripts/generate_data.py` first.


In [5]:
if 'e_id' in locals() and 'e_ood' in locals():
    plt.figure(figsize=(8, 4))
    plt.hist(np.log10(e_id + 1e-12), bins=30, alpha=0.6, label='ID', density=True)
    plt.hist(np.log10(e_ood + 1e-12), bins=30, alpha=0.6, label='OOD', density=True)
    plt.xlabel('log10(Relative L2 Error)')
    plt.ylabel('Density')
    plt.title('Error Distribution: ID vs OOD')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()